# Muziekclassificatie -- Ongesuperviseerd (GMM)

**Project R.E.M. -- Notebook 4 van 4**

## Doel

Laat de data zelf bepalen hoeveel muziekgroepen er zijn via **Gaussian Mixture Models**.
Dit notebook vergelijkt twee scenarios:
- **k=3 geforceerd** -- drie groepen, direct vergelijkbaar met calm/neutral/energy
- **k=optimaal** -- aantal clusters gekozen door BIC (laagste score = beste model)

**Onderzoeksvraag (RQ5, kern):** *Weerspiegelt de audio-featureruimte de calm/neutral/energy
driedeling, of ondersteunt de data een andere clusterstructuur?*

## Waarom GMM?

Gaussian Mixture Models modelleren elk cluster als een multivariate Gaussische verdeling.
Dit is flexibeler dan K-Means (dat ronde clusters aanneemt) en geeft probabilistische
toewijzingen (kans dat een nummer tot elk cluster behoort).

GMM met BIC-modelselectie is de standaardbenadering voor data-gedreven clustering
wanneer het aantal clusters niet vooraf bekend is.
Zie: [sklearn GaussianMixture](https://scikit-learn.org/stable/modules/mixture.html)

## Wetenschappelijke relevantie

De bevinding dat BIC k=10 verkiest boven k=3 (BIC-verschil: -5961) is betekenisvol:
de Spotify audio-featureruimte is niet netjes in drie groepen op te delen.
Dit motiveert toekomstig werk: de optimale clusterindeling teruggeven aan de
playlistgenerator in plaats van handmatige calm/neutral/energy-drempels.

## Gebruik

| Instelling | Betekenis |
|---|---|
| `REUSE_MODEL = False` | Fit nieuwe modellen (BIC sweep k=2..10, ~1 min) |
| `REUSE_MODEL = True` | Laad bestaande modellen vanuit `models/music_unsupervised/` |

**Vereiste:** voor alle deelnemers moet `data/playlists/{codename}/playlists_generated/combined.csv`
bestaan (gegenereerd door de spotify_cli.py pipeline).

In [ ]:
from __future__ import annotations

import json
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
print('Imports OK')

## Configuratie

In [ ]:
# REUSE_MODEL: True  -> laad bestaande modellen (snel)
#              False -> fit nieuwe modellen (BIC sweep k=2..10, ~1 min)
REUSE_MODEL = False

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

PROJECT_ROOT  = Path().resolve().parent.parent
PLAYLISTS_DIR = PROJECT_ROOT / 'data' / 'playlists'
OUTPUT_DIR    = PROJECT_ROOT / 'data' / 'analysis' / 'music_unsupervised'
MODELS_DIR    = PROJECT_ROOT / 'models' / 'music_unsupervised'

# Audio-kenmerken voor clustering
FEATURES = ['tempo', 'energy', 'valence', 'danceability', 'acousticness', 'loudness']

# Voorfilter-drempels (zelfde als supervised notebook)
SPEECHINESS_MAX = 0.66
LIVENESS_MAX    = 0.80

# GMM zoekbereik
K_MIN, K_MAX = 2, 10

# Donker thema (consistent met Shiny-app)
DARK = {
    'figure.facecolor': '#0f1218', 'axes.facecolor': '#181e2a',
    'axes.edgecolor': '#4a5568', 'axes.labelcolor': '#e2e8f0',
    'xtick.color': '#e2e8f0', 'ytick.color': '#e2e8f0',
    'text.color': '#e2e8f0', 'grid.color': '#2d3748', 'grid.alpha': 0.5,
    'font.family': 'monospace', 'legend.facecolor': '#1a2035',
    'legend.edgecolor': '#4a5568',
}
plt.rcParams.update(DARK)
plt.rcParams['figure.dpi'] = 120

# Okabe-Ito kleuren uitgebreid voor 10 clusters
OKABE_ITO = ['#56b4e9', '#e69f00', '#009e73', '#f0e442',
             '#0072b2', '#d55e00', '#cc79a7', '#999999', '#e2e8f0', '#7fcdbb']

print(f'PROJECT_ROOT : {PROJECT_ROOT}')
print(f'REUSE_MODEL  : {REUSE_MODEL}')

## 1. Data laden

Alle deelnemers worden samengenomen in een globaal model -- het GMM zoekt naar
patronen in de gehele muziekbibliotheek, niet per deelnemer. URI-deduplicatie
verwijdert nummers die meerdere deelnemers delen zodat ze niet dubbel tellen.

In [ ]:
def load_all_songs() -> pd.DataFrame:
    """Laad combined.csv van alle deelnemers en combineer."""
    frames = []
    for participant_dir in sorted(PLAYLISTS_DIR.iterdir()):
        if not participant_dir.is_dir():
            continue
        codename = participant_dir.name
        csv_path = participant_dir / 'playlists_generated' / 'combined.csv'
        if not csv_path.exists():
            continue
        df = pd.read_csv(csv_path)
        df['participant'] = codename
        frames.append(df)

    if not frames:
        raise FileNotFoundError(
            f'Geen combined.csv gevonden onder {PLAYLISTS_DIR}.\n'
            'Verwerk eerst de playlists met de spotify_cli.py pipeline.'
        )

    df_all = pd.concat(frames, ignore_index=True)

    if 'uri' in df_all.columns:
        n_before = len(df_all)
        df_all = df_all.drop_duplicates(subset='uri', keep='first')
        print(f'  URI-deduplicatie: {n_before - len(df_all)} duplicaten verwijderd')

    return df_all


def prefilter(df: pd.DataFrame) -> pd.DataFrame:
    """Verwijder live-opnames en gesproken-woord tracks."""
    n_before = len(df)
    mask = pd.Series(True, index=df.index)
    if 'speechiness' in df.columns:
        mask &= df['speechiness'] <= SPEECHINESS_MAX
    if 'liveness' in df.columns:
        mask &= df['liveness'] <= LIVENESS_MAX
    df = df[mask].copy()
    n_removed = n_before - len(df)
    if n_removed:
        print(f'  Voorfilter: {n_removed} tracks verwijderd (speech/live)')
    return df


df_raw = load_all_songs()
print(f"  Geladen: {len(df_raw)} nummers, {df_raw['participant'].nunique()} deelnemer(s)")

df = prefilter(df_raw)

missing_feats = [f for f in FEATURES if f not in df.columns]
if missing_feats:
    raise ValueError(f'Ontbrekende kolommen: {missing_feats}')
n_before = len(df)
df = df.dropna(subset=FEATURES).copy()
print(f'  Na NaN-verwijdering: {len(df)} nummers ({n_before - len(df)} verwijderd)')

## 2. Normalisatie

**Waarom StandardScaler (niet MinMaxScaler)?**
GMM veronderstelt Gaussische verdelingen. StandardScaler brengt alle kenmerken naar
gemiddelde 0 en standaarddeviatie 1, zodat geen enkel kenmerk het model domineert
door een groter numeriek bereik (bijv. tempo in BPM vs energy in 0-1).

Zie: [sklearn StandardScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html)

In [ ]:
MODELS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

scaler_path = MODELS_DIR / 'scaler.pkl'

if REUSE_MODEL:
    if not scaler_path.exists():
        raise FileNotFoundError(
            f'REUSE_MODEL=True maar scaler niet gevonden: {scaler_path}\n'
            'Zet REUSE_MODEL=False en voer het notebook opnieuw uit.'
        )
    scaler = joblib.load(scaler_path)
    print(f'Scaler geladen vanuit {scaler_path.name}')
else:
    scaler = StandardScaler()
    scaler.fit(df[FEATURES].values)
    joblib.dump(scaler, scaler_path)
    print(f'Scaler gefittet en opgeslagen -> {scaler_path.name}')

X = scaler.transform(df[FEATURES].values)
print(f'Feature-matrix: {X.shape}')

## 3. GMM-modelselectie -- BIC (k=2..10)

**BIC (Bayesian Information Criterion)** beloont modellen die de data goed verklaren
maar straft onnodig complexe modellen (te veel clusters):
```
BIC = -2 * log-likelihood + k * log(n)
```
De laagste BIC-score wijst het optimale aantal clusters aan.

**AIC (Akaike Information Criterion)** is vergelijkbaar maar straft complexiteit minder zwaar:
```
AIC = -2 * log-likelihood + 2k
```

Beide zijn informatiecriteria voor modelselectie. Als BIC en AIC een ander optimum geven,
is er geen duidelijke beste keuze -- meer data nodig.
Zie: [sklearn GaussianMixture BIC](https://scikit-learn.org/stable/modules/mixture.html#selecting-the-number-of-components)

In [ ]:
gmm_k3_path  = MODELS_DIR / 'gmm_k3.pkl'
gmm_opt_path = MODELS_DIR / 'gmm_opt.pkl'
config_path  = MODELS_DIR / 'config.json'

if REUSE_MODEL:
    for p in [gmm_k3_path, gmm_opt_path, config_path]:
        if not p.exists():
            raise FileNotFoundError(
                f'REUSE_MODEL=True maar bestand niet gevonden: {p}\n'
                'Zet REUSE_MODEL=False en voer het notebook opnieuw uit.'
            )
    gmm_k3  = joblib.load(gmm_k3_path)
    gmm_opt = joblib.load(gmm_opt_path)
    with open(config_path, encoding='utf-8') as f:
        config = json.load(f)
    optimal_k   = config['optimal_k']
    bic_results = config.get('bic_sweep', {})
    print(f'Modellen geladen. Optimaal k={optimal_k}')

else:
    print(f'GMMs fitten voor k={K_MIN}..{K_MAX}...')
    bic_results: dict = {}
    for k in range(K_MIN, K_MAX + 1):
        gmm = GaussianMixture(
            n_components=k, covariance_type='full',
            n_init=5, random_state=RANDOM_STATE,
        )
        gmm.fit(X)
        labels = gmm.predict(X)
        sil = silhouette_score(X, labels) if k > 1 else 0.0
        bic_results[k] = {
            'bic': gmm.bic(X), 'aic': gmm.aic(X),
            'silhouette': sil, 'model': gmm,
        }
        print(f'  k={k:2d}: BIC={gmm.bic(X):>10.0f}, silhouette={sil:.3f}')

    optimal_k = min(bic_results, key=lambda k: bic_results[k]['bic'])
    print(f'\n-> Optimaal k = {optimal_k} (laagste BIC)')

    gmm_k3  = bic_results[3]['model']
    gmm_opt = bic_results[optimal_k]['model']

    joblib.dump(gmm_k3,  gmm_k3_path)
    joblib.dump(gmm_opt, gmm_opt_path)
    print(f'Modellen opgeslagen -> {gmm_k3_path.name}, {gmm_opt_path.name}')

    config = {
        'optimal_k':      optimal_k,
        'features':       FEATURES,
        'n_songs':        len(df),
        'n_participants': df['participant'].nunique() if 'participant' in df.columns else 0,
        'k3_bic':         bic_results[3]['bic'],
        'k3_silhouette':  bic_results[3]['silhouette'],
        'opt_bic':        bic_results[optimal_k]['bic'],
        'opt_silhouette': bic_results[optimal_k]['silhouette'],
        'bic_sweep': {
            str(k): {'bic': v['bic'], 'aic': v['aic'], 'silhouette': v['silhouette']}
            for k, v in bic_results.items()
        },
    }
    with open(config_path, 'w', encoding='utf-8') as f:
        json.dump(config, f, indent=2)
    print(f'Config opgeslagen -> {config_path.name}')

## 4. Diagnostiek -- BIC & silhouette curve

**BIC** daalt als meer clusters de data beter verklaren (lagere BIC = betere fit).
**Silhouette** meet hoe goed een punt bij zijn eigen cluster past vs andere clusters
(1 = perfecte scheiding, 0 = overlappende clusters, -1 = foutief toegewezen).

Lage silhouette bij optimale BIC-k betekent dat de clusters statistisch onderscheidbaar
zijn (BIC), maar ruimtelijk sterk overlappen in de featureruimte.
Dit is een eigenschap van de Spotify audio-featureruimte, niet een modelfout.

In [ ]:
if REUSE_MODEL:
    sweep = config.get('bic_sweep', {})
    ks   = sorted(int(k) for k in sweep)
    bics = [sweep[str(k)]['bic']        for k in ks]
    aics = [sweep[str(k)]['aic']        for k in ks]
    sils = [sweep[str(k)]['silhouette'] for k in ks]
else:
    ks   = sorted(bic_results)
    bics = [bic_results[k]['bic']        for k in ks]
    aics = [bic_results[k]['aic']        for k in ks]
    sils = [bic_results[k]['silhouette'] for k in ks]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(ks, bics, 'o-', color='#56b4e9', label='BIC', linewidth=2)
ax1.plot(ks, aics, 's--', color='#e69f00', label='AIC', linewidth=1.5)
ax1.axvline(optimal_k, color='#cc79a7', linestyle=':', linewidth=2,
            label=f'Optimaal k={optimal_k}')
if optimal_k != 3:
    ax1.axvline(3, color='#009e73', linestyle=':', linewidth=1.5, alpha=0.7,
                label='k=3 (geforceerd)')
ax1.set_xlabel('Aantal clusters (k)')
ax1.set_ylabel('Informatiecriterium')
ax1.set_title('Modelselectie: BIC & AIC')
ax1.legend(fontsize=8)
ax1.grid(True, alpha=0.3)

ax2.plot(ks, sils, 'o-', color='#009e73', linewidth=2)
ax2.axvline(optimal_k, color='#cc79a7', linestyle=':', linewidth=2,
            label=f'Optimaal k={optimal_k}')
if optimal_k != 3:
    ax2.axvline(3, color='#009e73', linestyle=':', linewidth=1.5, alpha=0.7,
                label='k=3 (geforceerd)')
ax2.set_xlabel('Aantal clusters (k)')
ax2.set_ylabel('Silhouette-score')
ax2.set_title('Clusterkwaliteit: Silhouette')
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)

fig.suptitle('GMM modelselectie (k=2..10)', fontsize=13)
plt.tight_layout()
plt.show()

print(f"Optimaal k={optimal_k}  |  BIC={config['opt_bic']:.0f}  |  silhouette={config['opt_silhouette']:.3f}")
print(f"k=3 geforceerd        |  BIC={config['k3_bic']:.0f}  |  silhouette={config['k3_silhouette']:.3f}")

## 5. Classificatie & clusterprofielen

Wijs clusteretiketten en probabilistische kansen toe voor beide modellen.
De `confidence`-kolom geeft de maximale posterior kans: 1.0 = zeker, 0.5 = gelijke kans.

In [ ]:
df = df.copy()

for label, gmm, k in [('k3', gmm_k3, 3), ('opt', gmm_opt, optimal_k)]:
    probs = gmm.predict_proba(X)
    df[f'cluster_{label}']    = gmm.predict(X)
    df[f'confidence_{label}'] = probs.max(axis=1)

def cluster_profiles(df: pd.DataFrame, cluster_col: str) -> pd.DataFrame:
    return df.groupby(cluster_col)[FEATURES].agg(['mean', 'std']).round(3)

profiles_k3  = cluster_profiles(df, 'cluster_k3')
profiles_opt = cluster_profiles(df, 'cluster_opt')

print('Clusterprofiel k=3:')
display(profiles_k3)

print(f'\nClusterprofiel k={optimal_k} (optimaal):')
display(profiles_opt)

print('\nClusterverdeling per deelnemer (k=3):')
if 'participant' in df.columns:
    display(
        df.groupby(['participant', 'cluster_k3'])
        .size().unstack(fill_value=0)
        .rename(columns=lambda c: f'cluster {c}')
    )

## 6. Opslaan

In [ ]:
internal_cols = [c for c in df.columns if c.startswith('prob_')]
save_cols = [c for c in df.columns if c not in internal_cols]

csv_k3  = OUTPUT_DIR / 'classified_songs_k3.csv'
csv_opt = OUTPUT_DIR / f'classified_songs_k{optimal_k}.csv'

df[save_cols].to_csv(csv_k3,  index=False)
df[save_cols].to_csv(csv_opt, index=False)
print(f'  {csv_k3.name}')
print(f'  {csv_opt.name}')

profiles_k3.to_csv(OUTPUT_DIR / 'cluster_means_k3.csv')
profiles_opt.to_csv(OUTPUT_DIR / f'cluster_means_k{optimal_k}.csv')
print(f'  cluster_means_k3.csv')
print(f'  cluster_means_k{optimal_k}.csv')
print('\nKlaar.')

---

## 7. PCA-scatter -- k=3 vs optimaal

PCA reduceert de zes audio-kenmerken naar twee assen voor visualisatie.
Elk punt is een nummer, de kleur geeft het cluster aan.

Goed gescheiden kleurblokken = duidelijk onderscheiden muziekgroepen.
Sterke overlap (zoals hier bij hoge k) betekent dat de clusters statistisch
onderscheidbaar zijn maar ruimtelijk niet goed gescheiden -- typisch voor
rijke, hoge-dimensionale featureruimten.

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_2d = pca.fit_transform(X)
var_explained = pca.explained_variance_ratio_

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (label, k) in zip(axes, [('k3', 3), ('opt', optimal_k)]):
    cluster_col = f'cluster_{label}'
    for i in range(k):
        mask = df[cluster_col] == i
        color = OKABE_ITO[i % len(OKABE_ITO)]
        ax.scatter(
            X_2d[mask, 0], X_2d[mask, 1],
            c=color, label=f'Cluster {i}',
            alpha=0.45, s=12, edgecolors='none',
        )
    title = 'k=3 (geforceerd)' if label == 'k3' else f'k={optimal_k} (optimaal)'
    ax.set_title(title)
    ax.set_xlabel(f'PC1 ({var_explained[0]*100:.1f}%)')
    ax.set_ylabel(f'PC2 ({var_explained[1]*100:.1f}%)')
    ax.legend(fontsize=7, markerscale=2)
    ax.grid(True, alpha=0.3)

fig.suptitle('PCA-projectie van muziekclusters', fontsize=13)
plt.tight_layout()
plt.show()

## 8. Radar chart -- kenmerkprofiel per cluster (k=3)

Wat maakt elk cluster uniek? De radar toont de gemiddelde waarden per audio-kenmerk,
genormaliseerd over clusters voor vergelijkbaarheid.

Typische k=3-interpretatie: een cluster met hoge tempo + energy = 'energy',
een cluster met hoge acousticness + lage energy = 'calm', de derde is 'other'.
Of de GMM-clusters overeenkomen met deze labels is de kern van RQ5.

In [ ]:
radar_features = [f for f in FEATURES if f in df.columns]
radar_data = df.groupby('cluster_k3')[radar_features].mean()

# Normaliseer per kenmerk naar [0, 1] voor vergelijkbaarheid
for col in radar_data.columns:
    col_min, col_max = radar_data[col].min(), radar_data[col].max()
    if col_max > col_min:
        radar_data[col] = (radar_data[col] - col_min) / (col_max - col_min)

n_feat  = len(radar_features)
angles  = np.linspace(0, 2 * np.pi, n_feat, endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw={'projection': 'polar'})
ax.set_facecolor('#181e2a')

for idx, (cluster_id, row) in enumerate(radar_data.iterrows()):
    values = row.tolist() + row.tolist()[:1]
    color  = OKABE_ITO[idx % len(OKABE_ITO)]
    ax.plot(angles, values, 'o-', color=color, linewidth=2,
            label=f'Cluster {cluster_id}')
    ax.fill(angles, values, alpha=0.12, color=color)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(radar_features, fontsize=9)
ax.set_yticks([0.25, 0.5, 0.75])
ax.set_yticklabels(['0.25', '0.50', '0.75'], fontsize=6, color='#4a5568')
ax.tick_params(colors='#e2e8f0')
ax.set_title('Kenmerkprofiel per cluster -- k=3', pad=16)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), fontsize=9)
ax.grid(color='#2d3748', alpha=0.5)
plt.tight_layout()
plt.show()

## 9. Vergelijkingstabel -- k=3 vs optimaal

Is k=3 een verdedigbare keuze?

- **Lagere BIC** bij k=optimaal = de data ondersteunt meer dan 3 clusters statistisch.
- **Lagere silhouette** bij k=optimaal = de extra clusters zijn statistisch onderscheidbaar
  maar ruimtelijk minder goed gescheiden (elk cluster is smaller/meer overlappend).

**Conclusie voor het project:** k=3 is bruikbaar als directe vergelijking met calm/neutral/energy
en produceert beter gescheiden clusters (hogere silhouette). k=optimaal vertegenwoordigt
de fijnere structuur van de data en is interessant voor toekomstige verfijning van de
playlistgenerator.

In [ ]:
comparison = pd.DataFrame([
    {
        'Model':        'k=3 (geforceerd)',
        'k':            3,
        'BIC':          round(config['k3_bic']),
        'Silhouette':   round(config['k3_silhouette'], 3),
        'BIC-verschil': 0,
    },
    {
        'Model':        f'k={optimal_k} (optimaal)',
        'k':            optimal_k,
        'BIC':          round(config['opt_bic']),
        'Silhouette':   round(config['opt_silhouette'], 3),
        'BIC-verschil': round(config['opt_bic'] - config['k3_bic']),
    },
])
display(comparison.set_index('Model'))

print('\nClustergroottes k=3:')
display(df['cluster_k3'].value_counts().sort_index().rename('n_songs'))

print(f'\nClustergroottes k={optimal_k}:')
display(df['cluster_opt'].value_counts().sort_index().rename('n_songs'))

bic_diff = config['opt_bic'] - config['k3_bic']
if bic_diff < -10:
    print(f'\n-> BIC-verschil = {bic_diff:.0f}: k={optimal_k} past duidelijk beter bij de data dan k=3.')
elif bic_diff > 10:
    print(f'\n-> BIC-verschil = {bic_diff:.0f}: k=3 past verrassend goed -- data ondersteunt drie groepen.')
else:
    print(f'\n-> BIC-verschil = {bic_diff:.0f}: k=3 en k={optimal_k} zijn vergelijkbaar.')